In [13]:
import requests
import pandas as pd
import time
from datetime import datetime
from dotenv import load_dotenv
import os

In [14]:
load_dotenv() 
token = os.getenv('token')


In [15]:
group_name = 'ikanam'
api_version = '5.131'

all_posts = [] 
first_post = 0 
count = 100 

Получаем ID сообщества

In [16]:
url = f"https://api.vk.com/method/groups.getById?group_id={group_name}&access_token={token}&v={api_version}"
result = requests.get(url).json()
group_id = result['response'][0]['id']
group_id 

130344439

Запускаем бесконечный цикл: отправляется запрос к VK API и по итогу получаем словарь с ответами. Далее из результата надо получить список постов: 

если ключ 'response' есть, то возвращается словарь с данными, если его нет - то пустой словарь. И далее опять применяем функцию get, но теперь по ключу items, 

если его нет, то по итогу будет просто пустой список. 

Далее нужно проверить, пустой ли список с постами: если да, то это значит, что все посты обработаны и цикл завершается. И потом сравниваем длину полученного

списка и исходно заданного количества постов: если количество загруженных постов меньше, чем заданное значение, то новых постов больше нет, и цикл заканчивается. 

Между обработками делается пауза, потому что vk разрешает делать где-то 3 запроса в секунду, если паузу не делать, то можно перегрузить API запросами и будет 

блокировка за частые запросы

In [11]:
while True:
    result = requests.get('https://api.vk.com/method/wall.get',{
        'owner_id': -group_id,
        'count': count,
        'offset': first_post,
        'access_token': token,
        'v': api_version
    }).json()

    posts = result.get('response', {}).get('items', [])
    if not posts:
        break

    all_posts.extend(posts)
    
    if len(posts) < count:
        break

    first_post = first_post + count
    time.sleep(0.34)

Проходимся по всем постам и получем для каждого ID, дату публикации, текст поста и ссылку на этот пост. И сохраняем это всё в DataFrame

In [12]:
posts_data = []

for post in all_posts:
    post_id = post['id']
    date = datetime.fromtimestamp(post['date'])  
    text = post.get('text', '')
    source = f"https://vk.com/wall-{group_id}_{post_id}" 
    post_type = 'пост'
    for attachment in post.get('attachments', []):
        if attachment.get('type') == 'article':
            post_type = 'статья'
            break

    posts_data.append({
        'post_id': post_id,
        'date': date,
        'text': text,
        'source': source,
        'type': post_type
    })

df_posts = pd.DataFrame(posts_data)
df_posts

,post_id,date,text,source,type
0,5836,2025-03-25 22:30:00,"ХОБА, ВЕСЕННИЙ ЛОФТ!🎉\n\nУже чувствуете этот в...",https://vk.com/wall-130344439_5836,пост
1,5880,2025-05-27 12:29:30,Всем привет всем привет \n \nИщем датасаентист...,https://vk.com/wall-130344439_5880,пост
2,5879,2025-05-16 11:09:02,Всем общий саламчик!\n\nМы ищем в Мегафон DS у...,https://vk.com/wall-130344439_5879,пост
3,5878,2025-04-27 15:54:51,"Гаааайс, а вот и фотки с лофта подъехали📸🔥 \n ...",https://vk.com/wall-130344439_5878,пост
4,5875,2025-04-25 19:18:14,ДО ЛОФТА ЧАС🔥\n\nА пока вы готовитесь разносит...,https://vk.com/wall-130344439_5875,пост
...,...,...,...,...,...
1030,16,2016-10-10 01:25:31,Заливайте НОРМ мемы про ЕКАНАМ или учебу в аль...,https://vk.com/wall-130344439_16,пост
1031,7,2016-10-10 01:13:47,"Доброй ночи, эконом! Все мы знаем как сильна м...",https://vk.com/wall-130344439_7,пост
1032,5,2016-10-09 22:08:21,Сегодня последний день регистрации на киллера....,https://vk.com/wall-130344439_5,пост
1033,2,2016-10-09 21:53:07,,https://vk.com/wall-130344439_2,пост


Собираем комметарии. Создаём пустой список, чтобы добавлять туда комментарии. Запускаем цикл для каждого поста: получем ID для каждого поста и делаем 

запрос к vk API. И из ответа получаем список комментариев: проверяем, есть ли ключ 'response' в словаре, если нет — возвращаем пустой словарь,

затем из этого словаря получаем список с ключом 'items', если его нет, то список будет пустым. И далее перебираем каждый комментарий в этом списке, 

добавляем к каждому ID поста, ID комментария, дату написания комментария, текст и ссылку на пост. Результат представляем в DataFrame

In [6]:
comment_data = []

for i in range(len(df_posts['post_id'])):
    post_id = df_posts['post_id'][i]
    response = requests.get('https://api.vk.com/method/wall.getComments', {
        'owner_id': -group_id,
        'post_id': post_id,
        'count': 100,
        'access_token': token,
        'v': api_version
    }).json()
    
    comments = response.get('response', {}).get('items', [])
    
    for j in range(len(comments)):
        comment = comments[j]
        comment_data.append({
        'post_id': post_id,
        'comment_id': comment['id'],
        'date': datetime.fromtimestamp(comment['date']).strftime('%Y-%m-%d %H:%M'),
        'text': comment.get('text', ''),
        'source': f"https://vk.com/wall-{group_id}_{post_id}"
    })
    time.sleep(0.34)

df_comments = pd.DataFrame(comment_data)
df_comments

,post_id,comment_id,date,text,source
0,5851,5852,2025-04-13 00:19,"спасибо Гоше, что весь день жарил нам сосиски ❤🔥",https://vk.com/wall-130344439_5851
1,5845,5846,2025-04-10 22:05,Так неуютно спать без наблюдения кулера😭,https://vk.com/wall-130344439_5845
2,5842,5843,2025-04-09 22:07,"Ураааа, Ваня нажрался",https://vk.com/wall-130344439_5842
3,5842,5847,2025-04-10 22:11,Забыли упомянуть когда на частоте орги петь на...,https://vk.com/wall-130344439_5842
4,5818,5819,2025-02-11 20:43,Какому хачу,https://vk.com/wall-130344439_5818
...,...,...,...,...,...
2642,7,17,2016-10-10 02:40,у меня 1900,https://vk.com/wall-130344439_7
2643,2,3,2016-10-09 21:54,пикчер - бог,https://vk.com/wall-130344439_2
2644,2,4,2016-10-09 21:56,"[id6045249|Philip], +",https://vk.com/wall-130344439_2
2645,2,6,2016-10-09 22:27,В голос!,https://vk.com/wall-130344439_2


Объединяем таблицы с комментариями и с постами. По принципу: к постам добавляются комментарии, если пост был без комментариев, то под тем постом для комментария будет просто NA

In [23]:
df = df_posts.merge(df_comments, on='post_id', how='left')
df.rename(columns={
    'date_x': 'post_date',
    'text_x': 'post_text',
    'date_y': 'comment_date',
    'text_y': 'comment_text',
    'source_x': 'post_source',
    'source_y': 'comment_source'
}, inplace=True)
df


,post_id,post_date,post_text,post_source,type,comment_id,comment_date,comment_text,comment_source
0,5836,2025-03-25 22:30:00,"ХОБА, ВЕСЕННИЙ ЛОФТ!🎉\n\nУже чувствуете этот в...",https://vk.com/wall-130344439_5836,пост,NaN,NaN,NaN,NaN
1,5880,2025-05-27 12:29:30,Всем привет всем привет \n \nИщем датасаентист...,https://vk.com/wall-130344439_5880,пост,NaN,NaN,NaN,NaN
2,5879,2025-05-16 11:09:02,Всем общий саламчик!\n\nМы ищем в Мегафон DS у...,https://vk.com/wall-130344439_5879,пост,NaN,NaN,NaN,NaN
3,5878,2025-04-27 15:54:51,"Гаааайс, а вот и фотки с лофта подъехали📸🔥 \n ...",https://vk.com/wall-130344439_5878,пост,NaN,NaN,NaN,NaN
4,5875,2025-04-25 19:18:14,ДО ЛОФТА ЧАС🔥\n\nА пока вы готовитесь разносит...,https://vk.com/wall-130344439_5875,пост,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
3083,2,2016-10-09 21:53:07,,https://vk.com/wall-130344439_2,пост,3.0,2016-10-09 21:54,пикчер - бог,https://vk.com/wall-130344439_2
3084,2,2016-10-09 21:53:07,,https://vk.com/wall-130344439_2,пост,4.0,2016-10-09 21:56,"[id6045249|Philip], +",https://vk.com/wall-130344439_2
3085,2,2016-10-09 21:53:07,,https://vk.com/wall-130344439_2,пост,6.0,2016-10-09 22:27,В голос!,https://vk.com/wall-130344439_2
3086,2,2016-10-09 21:53:07,,https://vk.com/wall-130344439_2,пост,18.0,2016-10-10 07:53,Что це?,https://vk.com/wall-130344439_2


Колонку comment_source можно удалить, потому что ссылка на комментарий и пост идентична, так как комментарий под соответствующим постом. Заодно удаляем comment_date. 

Также если в comment_id стоит NA, то это значит, что пост был без комментария

In [24]:
df.drop(columns=['comment_date','comment_source'], inplace=True)
df['comment_id'].fillna('К посту комментариев нет', inplace=True)
df

/var/folders/p5/6n72_bgn12g00gw692k67yh00000gp/T/ipykernel_42498/3143365425.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['comment_id'].fillna('К посту комментариев нет', inplace=True)
/var/folders/p5/6n72_bgn12g00gw692k67yh00000gp/T/ipykernel_42498/3143365425.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'К посту комментариев нет' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df['commen

,post_id,post_date,post_text,post_source,type,comment_id,comment_text
0,5836,2025-03-25 22:30:00,"ХОБА, ВЕСЕННИЙ ЛОФТ!🎉\n\nУже чувствуете этот в...",https://vk.com/wall-130344439_5836,пост,К посту комментариев нет,NaN
1,5880,2025-05-27 12:29:30,Всем привет всем привет \n \nИщем датасаентист...,https://vk.com/wall-130344439_5880,пост,К посту комментариев нет,NaN
2,5879,2025-05-16 11:09:02,Всем общий саламчик!\n\nМы ищем в Мегафон DS у...,https://vk.com/wall-130344439_5879,пост,К посту комментариев нет,NaN
3,5878,2025-04-27 15:54:51,"Гаааайс, а вот и фотки с лофта подъехали📸🔥 \n ...",https://vk.com/wall-130344439_5878,пост,К посту комментариев нет,NaN
4,5875,2025-04-25 19:18:14,ДО ЛОФТА ЧАС🔥\n\nА пока вы готовитесь разносит...,https://vk.com/wall-130344439_5875,пост,К посту комментариев нет,NaN
...,...,...,...,...,...,...,...
3083,2,2016-10-09 21:53:07,,https://vk.com/wall-130344439_2,пост,3.0,пикчер - бог
3084,2,2016-10-09 21:53:07,,https://vk.com/wall-130344439_2,пост,4.0,"[id6045249|Philip], +"
3085,2,2016-10-09 21:53:07,,https://vk.com/wall-130344439_2,пост,6.0,В голос!
3086,2,2016-10-09 21:53:07,,https://vk.com/wall-130344439_2,пост,18.0,Что це?


Получаем статьи из раздела Публикации, это всё делаем вручную и сохраняем в файл exel

In [25]:
df_articles = pd.read_excel("/Users/alina/Desktop/Статьи.xlsx")
df_articles['date'] = pd.to_datetime(df_articles['date'], errors='coerce')
df_articles

,post_id,date,text,source,type
0,NaN,2022-08-02,"Первое научное руководство первакам N года, гд...",https://vk.com/@ikanam-pervoe-nauchnoe,статья
1,NaN,2023-04-14,Культура проведения nlp-хакатонов в Восточной ...,https://vk.com/@ikanam-culture-nlp-hackathons,статья
2,NaN,2023-08-07,Инструкция по жизни и выживанию на отделении э...,https://vk.com/@ikanam-instrukciya-po-zhizni-i...,статья
3,NaN,2022-11-30,Самоучитель для физиков и лириков эпоху переме...,https://vk.com/@ikanam-samouchitel,статья
4,NaN,2018-06-18,Как стать дата саентистом\nВнимание! Этот гайд...,https://vk.com/@ikanam-kak-stat-data-saentistom,статья
5,NaN,2018-06-30,Просто откройте и прочитайте\nМы хотим сделать...,https://vk.com/@ikanam-prosto-otkroite-i-proch...,статья
6,NaN,2018-04-23,Мини-карьерная пятница\nВсем привет! Тут 2 инт...,https://vk.com/@ikanam-mini-karernaya-pyatnica,статья
7,NaN,2018-03-05,"Сезон карьеры объявляется открытым\nПривет, вс...",https://vk.com/@ikanam-sezon-karery-obyavlyaet...,статья
8,NaN,2018-01-29,"Ответы на вопросы\nЧто делать сейчас мне, перв...",https://vk.com/@ikanam-otvety-na-voprosy,статья


Всё приводим к красивому виду, таблицы объединяем, если к посту комментария нет, то NA заменяем на 'К посту комментариев нет', то же самое и с ID. И поскольку у статей комментариев нет, то тоже пишем, что 'К посту комментариев нет'

In [28]:
df_2 = pd.DataFrame(columns=df.columns)
df_2['post_date'] = df_articles['date']
df_2['post_text'] = df_articles['text']
df_2['post_source'] = df_articles['source']
df_2['type'] = 'статья'
df_3 = pd.concat([df, df_2], ignore_index=True)
df_3.loc[df_3['comment_id'] == 'К посту комментариев нет', 'comment_text'] = 'К посту комментариев нет'
df_3.loc[df_3['type'] == 'статья', 'post_id'] = 'для статьи используем ссылку на пост'
df_3.loc[df_3['type'] == 'статья', 'comment_id'] = 'К посту комментариев нет'
df_3.loc[df_3['type'] == 'статья', 'comment_text'] = 'К посту комментариев нет'
df_3['post_text'].fillna('Пост состоит из фото', inplace=True)
df_3

/var/folders/p5/6n72_bgn12g00gw692k67yh00000gp/T/ipykernel_42498/3538463889.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_3['post_text'].fillna('Пост состоит из фото', inplace=True)


,post_id,post_date,post_text,post_source,type,comment_id,comment_text
0,5836,2025-03-25 22:30:00,"ХОБА, ВЕСЕННИЙ ЛОФТ!🎉\n\nУже чувствуете этот в...",https://vk.com/wall-130344439_5836,пост,К посту комментариев нет,К посту комментариев нет
1,5880,2025-05-27 12:29:30,Всем привет всем привет \n \nИщем датасаентист...,https://vk.com/wall-130344439_5880,пост,К посту комментариев нет,К посту комментариев нет
2,5879,2025-05-16 11:09:02,Всем общий саламчик!\n\nМы ищем в Мегафон DS у...,https://vk.com/wall-130344439_5879,пост,К посту комментариев нет,К посту комментариев нет
3,5878,2025-04-27 15:54:51,"Гаааайс, а вот и фотки с лофта подъехали📸🔥 \n ...",https://vk.com/wall-130344439_5878,пост,К посту комментариев нет,К посту комментариев нет
4,5875,2025-04-25 19:18:14,ДО ЛОФТА ЧАС🔥\n\nА пока вы готовитесь разносит...,https://vk.com/wall-130344439_5875,пост,К посту комментариев нет,К посту комментариев нет
...,...,...,...,...,...,...,...
3092,для статьи используем ссылку на пост,2018-06-18 00:00:00,Как стать дата саентистом\nВнимание! Этот гайд...,https://vk.com/@ikanam-kak-stat-data-saentistom,статья,К посту комментариев нет,К посту комментариев нет
3093,для статьи используем ссылку на пост,2018-06-30 00:00:00,Просто откройте и прочитайте\nМы хотим сделать...,https://vk.com/@ikanam-prosto-otkroite-i-proch...,статья,К посту комментариев нет,К посту комментариев нет
3094,для статьи используем ссылку на пост,2018-04-23 00:00:00,Мини-карьерная пятница\nВсем привет! Тут 2 инт...,https://vk.com/@ikanam-mini-karernaya-pyatnica,статья,К посту комментариев нет,К посту комментариев нет
3095,для статьи используем ссылку на пост,2018-03-05 00:00:00,"Сезон карьеры объявляется открытым\nПривет, вс...",https://vk.com/@ikanam-sezon-karery-obyavlyaet...,статья,К посту комментариев нет,К посту комментариев нет


Получаем датасет


In [29]:
df_3.to_csv("/Users/alina/Desktop/готовый датасет.csv", index=False, encoding='utf-8-sig')
df_3

,post_id,post_date,post_text,post_source,type,comment_id,comment_text
0,5836,2025-03-25 22:30:00,"ХОБА, ВЕСЕННИЙ ЛОФТ!🎉\n\nУже чувствуете этот в...",https://vk.com/wall-130344439_5836,пост,К посту комментариев нет,К посту комментариев нет
1,5880,2025-05-27 12:29:30,Всем привет всем привет \n \nИщем датасаентист...,https://vk.com/wall-130344439_5880,пост,К посту комментариев нет,К посту комментариев нет
2,5879,2025-05-16 11:09:02,Всем общий саламчик!\n\nМы ищем в Мегафон DS у...,https://vk.com/wall-130344439_5879,пост,К посту комментариев нет,К посту комментариев нет
3,5878,2025-04-27 15:54:51,"Гаааайс, а вот и фотки с лофта подъехали📸🔥 \n ...",https://vk.com/wall-130344439_5878,пост,К посту комментариев нет,К посту комментариев нет
4,5875,2025-04-25 19:18:14,ДО ЛОФТА ЧАС🔥\n\nА пока вы готовитесь разносит...,https://vk.com/wall-130344439_5875,пост,К посту комментариев нет,К посту комментариев нет
...,...,...,...,...,...,...,...
3092,для статьи используем ссылку на пост,2018-06-18 00:00:00,Как стать дата саентистом\nВнимание! Этот гайд...,https://vk.com/@ikanam-kak-stat-data-saentistom,статья,К посту комментариев нет,К посту комментариев нет
3093,для статьи используем ссылку на пост,2018-06-30 00:00:00,Просто откройте и прочитайте\nМы хотим сделать...,https://vk.com/@ikanam-prosto-otkroite-i-proch...,статья,К посту комментариев нет,К посту комментариев нет
3094,для статьи используем ссылку на пост,2018-04-23 00:00:00,Мини-карьерная пятница\nВсем привет! Тут 2 инт...,https://vk.com/@ikanam-mini-karernaya-pyatnica,статья,К посту комментариев нет,К посту комментариев нет
3095,для статьи используем ссылку на пост,2018-03-05 00:00:00,"Сезон карьеры объявляется открытым\nПривет, вс...",https://vk.com/@ikanam-sezon-karery-obyavlyaet...,статья,К посту комментариев нет,К посту комментариев нет
